In [ ]:
# importing libraries
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack

In [ ]:
df = pd.read_csv(r"C:\Users\deva\OneDrive\Desktop\My Project\spam mail.csv")
# Remove all unnamed columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
print("Successful")

In [ ]:
df.head()

In [ ]:
df.rename(columns={'Category':'label','Masseges':'message'}, inplace=True)
df.columns


In [ ]:
print("Dataset Shape:", df.shape)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['label'].value_counts()

In [ ]:
# Convert labels into numeric form
# Machine learning needs numbers.
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label_num'] = encoder.fit_transform(df['label'])

df.head()

In [ ]:
encoder.classes_

In [ ]:
# Calculates the no. of characters in the message
df['char_length'] = df['message'].apply(len)
print(df[['message', 'char_length']].head())

In [ ]:
#splits the message into words.
# counts the number of words in the message.
df['word_count'] = df['message'].apply(lambda x: len(x.split()))
print(df[['message', 'word_count']].head())

In [ ]:
# counts how many periods,  exclamation marks, question marks are in the message.
# Sum of these ≈ number of sentences (rough estimate).

df['sentence_count'] = df['message'].apply(
    lambda x: x.count('.') + x.count('!') + x.count('?')
)
print(df[['message', 'sentence_count']].head())

In [ ]:
important_phrases = [
    "verify account", "update bank", "click here",
    "limited time offer", "buy now", "free trial",
    "congratulations", "claim prize", "urgent response"
]

def count_suspicious(text):
    text = text.lower() # Convert text to lowercase for case-insensitive matching
    matches = [phrase for phrase in important_phrases if phrase in text]   # Find all phrases present
    print("Text:", text)
    print("Matched phrases:", matches)
    print("Count:", len(matches))
    print("-" * 50)
    return len(matches)   # Return number of suspicious phrases found
    
df['suspicious_phrase_count'] = df['message'].apply(count_suspicious)

In [ ]:
print(df['suspicious_phrase_count'].describe())

In [ ]:
print(df['suspicious_phrase_count'].value_counts())

In [ ]:
print(df.groupby('label')['suspicious_phrase_count'].mean())

In [ ]:
# URL Count
df['url_count'] = df['message'].apply(
    lambda x: len(re.findall(r'http[s]?://', x))
)

# Digit Frequency
df['digit_count'] = df['message'].apply(
    lambda x: len(re.findall(r'\d', x))
)

# Uppercase Pattern
df['uppercase_count'] = df['message'].apply(
    lambda x: sum(1 for word in x.split() if word.isupper())
)

# Special Character Usage
df['special_char_count'] = df['message'].apply(
    lambda x: len(re.findall(r'[!@#$%^&*(),.?":{}|<>]', x))
)

df.groupby('label')[['url_count','digit_count',
                     'uppercase_count','special_char_count']].mean()

In [ ]:
def clean_text(text):
    text = text.lower()   # Convert to lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text)   # Remove everything except letters, numbers, and spaces
    return text

df['cleaned_message'] = df['message'].apply(clean_text)

print(df[['message', 'cleaned_message']].sample(5, random_state=42)) 

In [ ]:
tfidf = TfidfVectorizer(
    analyzer='char',    # Analyze text by characters instead of words
    ngram_range=(2,5),   # Use character n-grams of length 2 to 5
    min_df=2,   # Ignore n-grams that appear in fewer than 2 messages
    max_features=115583    # Limit the number of features to avoid memory overload
)

X_text = tfidf.fit_transform(df['cleaned_message'])

print("Shape:", X_text.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))

In [ ]:
df['cleaned_message'] = df['message'].apply(clean_text)

df['char_length'] = df['cleaned_message'].apply(len)
df['word_count'] = df['cleaned_message'].apply(lambda x: len(x.split()))
df['sentence_count'] = df['cleaned_message'].apply(lambda x: len(re.findall(r'[.!?]+', x)))
df['url_count'] = df['cleaned_message'].apply(lambda x: len(re.findall(r'http[s]?://', x)))  # number of links.
df['digit_count'] = df['cleaned_message'].apply(lambda x: sum(c.isdigit() for c in x))   # number of digits (OTP, amounts).
df['uppercase_count'] = df['message'].apply(lambda x: sum(c.isupper() for c in x))   # fully capitalized words.
df['special_char_count'] = df['message'].apply(lambda x: len(re.findall(r'[^\w\s]', x)))  # punctuation and symbols.
phish_phrases = [
    "verify account", "update bank", "click here",
    "confirm identity", "password reset", "refund pending",
    "bank alert", "update account", "account suspended",
    "unusual activity", "security alert", "reset password",
    "click the link", "login immediately", "bank details",
    "credit card issue", "otp verification", "claim your prize"
]        # List of phishing-related phrases

spam_phrases = [
    "limited time offer", "buy now", "free trial",
    "exclusive deal", "don't miss out", "mega sale",
    "get discount", "70% off", "coupon code",
    "flash sale", "apply code now", "congratulations you won",
    "winner", "free gift", "last chance",
    "click here immediately", "limited stock",
    "download today", "trial expires",
    "shop today", "order now", "urgent message",
    "flash offer"
]       # List of spam-related phrases

df['phish_score'] = df['cleaned_message'].apply(
    lambda x: 2 * sum(p in x for p in phish_phrases)
)  # Counts how many phishing phrases appear in each message.

df['spam_score'] = df['cleaned_message'].apply(
    lambda x: 2 * sum(p in x for p in spam_phrases)
)   # Counts how many spam phrases appear in each message.

behaviour_features = np.column_stack([          # combines all columns into a single 2D NumPy array.
    df['char_length'],
    df['word_count'],
    df['sentence_count'],
    df['url_count'],
    df['digit_count'],
    df['uppercase_count'],
    df['special_char_count'],
    df['phish_score'],
    df['spam_score']
])

print("Final Behaviour Features Shape:", behaviour_features.shape)

In [ ]:
scaler = MinMaxScaler()
behaviour_scaled = scaler.fit_transform(behaviour_features)

print("Min value:", behaviour_scaled.min())
print("Max value:", behaviour_scaled.max())

In [ ]:
X = hstack((X_text, behaviour_scaled))
print("Final X shape:",X.shape)
y = df['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,  # Features and target
    test_size=0.2,  # 20% of data for testing, 80% for training
    random_state=42, # Ensures you get the same split every time (# reproducibility)
    stratify=y   # Keeps label distribution same in train and test sets
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
model = MultinomialNB(alpha=0.1)
model.fit(X_train, y_train)

In [ ]:
df['predicted_label'] = model.predict(X)

In [ ]:
from sklearn.metrics import confusion_matrix
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)   # compares true labels (y_test) with predicted labels (y_pred).

print(cm)

In [ ]:
print(df[['label', 'predicted_label']].head())

In [ ]:
# Check how many input features the trained model expects
print("Model expects:",model.n_features_in_)

In [ ]:
df[['message','label','predicted_label']].head(10)

In [ ]:
# Import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the confusion matrix as a heatmap
sns.heatmap(cm, annot=True, fmt='d')  
# annot=True -> show numbers inside the cells
# fmt='d' -> format numbers as integers

# Label the axes for clarity
plt.xlabel("Predicted")  # x-axis = predicted labels
plt.ylabel("Actual")     # y-axis = true labels

plt.show()  # Display the heatmap

In [ ]:
# Find wrong predictions
wrong = df[df['label'] != df['predicted_label']]

wrong.head(10)

In [ ]:
wrong.shape

In [ ]:
wrong['label'].value_counts()

In [ ]:
df.to_excel(r"C:\Users\deva\OneDrive\Desktop\My Project\final_analysis.xlsx", index=False)
print("file saved")

In [ ]:
wrong.to_excel(r"C:\Users\deva\OneDrive\Desktop\My Project\wrong_predictions.xlsx", index=False)
print("wrong_predictions file also saved")

In [ ]:
def extract_behaviour_features(text):
    return [
        len(text),
        len(text.split()),
        text.count('.') + text.count('!') + text.count('?'),
        count_suspicious(text),
        len(re.findall(r'http[s]?://', text)),
        len(re.findall(r'\d', text)),
        sum(1 for word in text.split() if word.isupper()),
        len(re.findall(r'[!@#$%^&*(),.?":{}|<>]', text))
    ]

In [ ]:
pickle.dump(model, open("spam_model_v3.pkl", "wb"))  # Save the trained Naive Bayes model to a file
pickle.dump(tfidf, open("tfidf_vectorizer_v3.pkl", "wb"))  # Save the TF-IDF vectorizer to a file
pickle.dump(scaler, open("scaler_v3.pkl", "wb"))  # Save the MinMax scaler to a file

print("Models saved with exact feature alignment")

In [ ]:
def predict_email(text):
    cleaned = clean_text(text)   # Clean the text
    text_vector = tfidf.transform([cleaned])   # Convert text to TF-IDF vector
    
    behaviour = np.array(extract_behaviour_features(text)).reshape(1, -1)
    behaviour_scaled = scaler.transform(behaviour)   # Scale behavioral features
    
    final_input = hstack((text_vector, behaviour_scaled))   # Combine TF-IDF + behavioral
    
    prediction = model.predict(final_input)    # Predict label
    probability = model.predict_proba(final_input)   # Get probabilities for each class
    
    return prediction[0], probability

In [ ]:
import os
os.chdir(r"C:\Users\deva\OneDrive\Desktop\My Project")
print(os.getcwd())

In [ ]:
print("Dataset Shape:", df.shape)    # After using Machine Learning
df.columns

In [ ]:
# Convert labels into numeric form
# Machine learning needs numbers.
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label_num'] = encoder.fit_transform(df['label'])

df.head()

In [ ]:
accuracy = model.score(X_test, y_test)
print("Final Test Accuracy:",accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
df.groupby('label')['char_length'].max()

In [ ]:
df.groupby('label')['char_length'].min()

In [ ]:
df['prediction_status'] = (df['label'] == df['predicted_label']).astype(int)
df.groupby('label')['prediction_status'].mean()

In [ ]:
#total correct and wrong 
df['prediction_status'] = df['label'] == df['predicted_label']

df['prediction_status'].value_counts()

In [ ]:
# longest each
df[df['label']=='ham'].sort_values('char_length', ascending=False).head(1)

In [ ]:
df[df['label']=='spam'].sort_values('char_length', ascending=False).head(1)

In [ ]:
df[df['label']=='phish'].sort_values('char_length', ascending=False).head(1)

In [ ]:
print(df['suspicious_phrase_count'].value_counts())

In [ ]:
# top ham words
from collections import Counter

ham_words = ' '.join(df[df['label']=='ham']['cleaned_message']).split()

Counter(ham_words).most_common(20)

In [ ]:
# top spam words
from collections import Counter

spam_words = ' '.join(df[df['label']=='spam']['cleaned_message']).split()

Counter(spam_words).most_common(20)

In [ ]:
# top phishing words
from collections import Counter

phish_words = ' '.join(df[df['label']=='phish']['cleaned_message']).split()

Counter(phish_words).most_common(10)

In [ ]:
df.groupby('label')[['suspicious_phrase_count', 'url_count','digit_count',
                     'uppercase_count','special_char_count']].mean()

In [ ]:
import matplotlib.pyplot as plt

df[df['label']=='ham']['word_count'].hist()
plt.title("Ham Word Count")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df[df['label']=='spam']['word_count'].hist()
plt.title("Spam Word Count")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df[df['label']=='phish']['word_count'].hist()
plt.title("Phish Word Count")
plt.show()

In [ ]:
df[df['prediction_status'] == False]

In [ ]:
df['prediction_status'].value_counts(normalize=True) * 100

In [ ]:
df.groupby('label')[['spam_score', 'phish_score']].mean()

In [ ]:
df.to_excel(r"C:\Users\deva\OneDrive\Desktop\My Project\final_analysis_updated.xlsx", index=False)
print("file saved")

In [ ]:
!pip install mysql-connector-python pymysql sqlalchemy

import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Load cleaned file
df = pd.read_excel(r"C:\Users\deva\OneDrive\Desktop\My Project\final_analysis_updated.xlsx")

# MySQL connection
username = "root"
password = "root"
host = "127.0.0.1"
port = "3306"
database = "spam_model"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# Upload to MySQL
df.to_sql("spam_model_messages", engine, if_exists="replace", index=False)

# Verify upload
pd.read_sql("SELECT * FROM spam_model_messages LIMIT 5;", engine)